In [ ]:
import torch

from transformers import T5EncoderModel
from transformers import AutoTokenizer
ckpt_path = '../runs/diversity/with_tac/2024_07_05/16_34/checkpoints/tac_enc'


In [ ]:
ckpt = torch.load(ckpt_path)

In [ ]:
state_dict = {k[12:]: v for k, v in ckpt.items() if k.startswith('tac_encoder')}

In [ ]:

tac_encoder = T5EncoderModel.from_pretrained('sean-lamont/leandojo-lean3-reprover-novel-premises',
                                             state_dict=state_dict).cuda()





tokenizer = AutoTokenizer.from_pretrained('sean-lamont/leandojo-lean3-reprover-novel-premises')


In [ ]:
import pickle

test_trace = "../runs/bestfs-novel-2/affine_subspace.is_connected_set_of_s_opp_side"
trace = pickle.load(open(test_trace, 'rb'))

In [ ]:

import torch.nn.functional as F

from torchmetrics.functional import pairwise_cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

def _encode(
        encoder, input_ids: torch.LongTensor, attention_mask: torch.LongTensor
) -> torch.FloatTensor:
    hidden_states = encoder(
        input_ids=input_ids,
        attention_mask=attention_mask,
        return_dict=True,
    ).last_hidden_state

    # Masked average.
    lens = attention_mask.sum(dim=1)
    features = (hidden_states * attention_mask.unsqueeze(2)).sum(
        dim=1
    ) / lens.unsqueeze(1)

    # Normalize the feature vector to have unit norm.
    return F.normalize(features, dim=1)


In [ ]:

node = trace.tree

tacs = [e.tactic for e in node.out_edges]
with torch.no_grad():
    tokenized_tacs = [tokenizer( e,
                                padding="longest",
                                max_length=2300,
                                truncation=True,
                                return_tensors="pt") for e in tacs]

    vecs = [_encode(tac_encoder, tac.input_ids.cuda(), tac.attention_mask.cuda())[0] for tac in tokenized_tacs]
    vec_matrix = torch.stack(vecs)

    cosine_sim = pairwise_cosine_similarity(vec_matrix)


In [ ]:

start = 20 
end = 35

# Convert the tensor to a NumPy array
similarity_matrix_np = cosine_sim.cpu().detach().numpy()[start:end, start:end]

names = tacs[start:end]
# Create the heatmap
plt.figure(figsize=(10, 8))
ax = sns.heatmap(similarity_matrix_np, xticklabels=names, yticklabels=names, annot=True, cmap='viridis')

# Move x-axis labels to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

# Slant x-axis labels
plt.xticks(rotation=45, ha='left')

# Add labels and title
plt.xlabel('Vectors')
plt.ylabel('Vectors')
plt.title('Pairwise Cosine Similarity Heatmap')

# Display the heatmap
plt.show()